# TTS Dataset Generation

Reads text prompts and writes a manifest with synthetic speech from one speaker, in the format
[docs/manifest.md](../docs/manifest.md) defines. This notebook is steps 1 and 2a of the chain
described in [docs/pipeline.md](../docs/pipeline.md): it turns a prompts source into the text
rows of a manifest, then synthesises each row with a single donor voice. Voice conversion,
measurement and selection are later, separate notebooks
([tts_voice_convert.ipynb](tts_voice_convert.ipynb), [tts_measure.ipynb](tts_measure.ipynb),
[tts_select.ipynb](tts_select.ipynb)); this notebook only synthesises.

The default donor is [OmniVoice](https://github.com/k2-fsa/OmniVoice), a multilingual
(646 languages) zero-shot speech model that synthesises and clones a voice in one step on a GPU.
A training set needs one speaker, so the notebook fixes one voice for the whole run: it clones the
clip in `REF_AUDIO`, or designs a voice from `VOICE_PROMPT` (a short description such as
`female, young adult, moderate pitch`) and clones that, or, with neither set, uses a fixed default
design. When `TTS_LANG` is not one of the 646 languages, the notebook stops and names the other
route: `DONOR_MODEL`, a model from the [phoonnx](https://pypi.org/project/phoonnx/) catalog or an
OVOS TTS plugin.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the prompts source, the voice and every parameter via env vars |
| 1 | Load prompts | Read text from `PROMPTS_FILE`, `PROMPTS_DIR` or `PROMPTS_CSV` (or the built-in demo prompts); refuse empty, bracketed or letterless rows |
| 2 | Donor synthesis | Load OmniVoice, fix one voice, synthesise every kept prompt in batches |
| 3 | Post-processing | Resample to `TARGET_SR`, mono, silence trim, loudness normalisation, and an optional `audiosronnx` super-resolution stage |
| 4 | Manifest, sidecar and summary | Write `manifest.jsonl`, its `.meta.json`, `metadata.csv`, and print the counts, the generation rate and the GPU memory used |

### Output structure
```
OUTPUT_DIR/
  voice/                # the fixed voice: anchor.wav and anchor.txt (designed voice) or the
                         # fetched reference clip, so a later run can continue the same speaker
  donor_raw/            # the donor's own raw audio per kept prompt, before post-processing
  wavs/                 # the final, post-processed WAVs (manifest.jsonl points here)
  manifest.jsonl
  manifest.jsonl.meta.json
  metadata.csv          # LJSpeech-style "<id>|<text>", for convenience
```

| Field | Value |
|---|---|
| Inputs | A prompts source: `PROMPTS_FILE` (one sentence per line, or piper-recording-studio `id<TAB>text`), `PROMPTS_DIR` (a folder of `.txt` files), or `PROMPTS_CSV` (a CSV/TSV with a text column); set exactly one, or leave all empty for the built-in demo prompts (English, Spanish, Portuguese, French, German, Italian). The voice: `REF_AUDIO` (a path or URL) with an optional `REF_TEXT`, or `VOICE_PROMPT`, or neither. |
| Outputs | `manifest.jsonl` (one row per synthesised prompt: `id`, `text`, `lang`, `audio`, `duration_s`, `sample_rate`, `source`, per `docs/manifest.md`), `manifest.jsonl.meta.json` (the donor, the voice, the counts, the refusal reasons, the generation rate and the GPU memory), and `metadata.csv` (the same rows as `<audio filename>|<text>`, for tools that read LJSpeech). |
| Runtime | On a Kaggle T4: the install and weight download, then about 25 s to load the model, then about 0.4 clips per second (2 s of audio per second). The 12 built-in demo prompts take about 2 minutes in all; 1,000 prompts take about 40 minutes. |
| Hardware | A GPU for OmniVoice (a Kaggle or Colab T4 is enough; it runs in fp16). A `DONOR_MODEL` donor and the CI path run on a CPU. |
| Cost | None: no paid API. The notebook needs outbound network access to install `omnivoice` and download its weights. |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path. CI has no GPU, so
`SMOKE=True` replaces OmniVoice with `espeak-ng`, a CPU synthesiser that exists for CI only and is not
a donor to build a dataset from.

**Read more:**

- [OmniVoice](https://github.com/k2-fsa/OmniVoice)
- [OVOS TTS plugins](https://openvoiceos.github.io/ovos-technical-manual/320-tts_plugins/)
- [phoonnx](https://github.com/TigreGotico/phoonnx)
- [synthetic_dataset_generator](https://github.com/TigreGotico/synthetic_dataset_generator), the pipeline this notebook continues

You can find `.txt` prompts here:
  - https://huggingface.co/datasets/fdemelo/phonetic-piper-recording-studio-prompts
  - https://huggingface.co/collections/TigreGotico/synthetic-tts-datasets
  - https://github.com/rhasspy/piper-recording-studio/tree/master/prompts

In [ ]:
# Parameters
SMOKE = False   # True: the CI path. No GPU: espeak-ng replaces OmniVoice, and the built-in prompts are mixed with rows that must be refused
N_SAMPLES = 12  # how many of the built-in demo prompts to use when no prompts source is set (at most 12)

> ⚠ **The default run needs a GPU and outbound network access.** On Kaggle: Settings, Accelerator,
> GPU T4, and Internet on. The notebook installs `omnivoice` and downloads its weights from
> Hugging Face on the first run. Without a GPU the notebook stops before it loads anything.

---
## 0 · Configuration

All parameters are set as environment variables via `os.environ.setdefault()`. Override any of
them from your shell, a `.env` file, or `%env` magic **before** running the config cell. With
nothing set, the notebook synthesises its built-in English demo prompts in one designed voice.

The pipeline and the manifest format are described in [docs/pipeline.md](../docs/pipeline.md) and
[docs/manifest.md](../docs/manifest.md).

| Variable | Meaning |
|---|---|
| `TTS_LANG` | Language of the prompts: a BCP-47 tag (`en`, `es`, `pt-PT`) or an OmniVoice language name. The primary subtag selects the OmniVoice language |
| `REF_AUDIO`, `REF_TEXT` | A clip to clone (path or URL, 3 to 10 s of one clean speaker) and its transcript. `REF_TEXT` is required with `REF_AUDIO`: the notebook never transcribes a clip itself |
| `VOICE_PROMPT` | A voice description in OmniVoice's vocabulary, for example `male, middle-aged, low pitch, british accent`. Gender, age, pitch, a whispered style and (English only) accent are accepted |
| `DONOR_MODEL` | Optional. A voice id from the phoonnx catalog or the name of an OVOS TTS plugin; it replaces OmniVoice. Nothing else in the notebook lists models |
| `SEED` | Seed for the voice design and the generation |
| `BATCH_SIZE` | Clips per OmniVoice call |

**Edit the values below to match your setup, then run the cell.**

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


def _default_output_dir():
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return "/kaggle/working/tts_dataset_gen_out"
    if os.path.isdir("/content"):
        return "/content/tts_dataset_gen_out"
    return "./tts_dataset_gen_out"


# ──────────────────────────────────────────────
# GENERAL
# ──────────────────────────────────────────────
os.environ.setdefault("TTS_LANG",     "en")                    # BCP-47 tag, e.g. en, es, pt-PT; OmniVoice reads the primary subtag
os.environ.setdefault("OUTPUT_DIR",   _default_output_dir())   # manifest, wavs/ and metadata.csv are written here
os.environ.setdefault("MANIFEST_NAME", "manifest.jsonl")
os.environ.setdefault("PROMPTS_SOURCE_NAME", "")                # source.name for ids; defaults to the prompts file/folder's own name

# ──────────────────────────────────────────────
# PROMPTS  (Stage 1) — set at most one of these three; none = the built-in demo prompts
# ──────────────────────────────────────────────
os.environ.setdefault("PROMPTS_FILE", "")   # one sentence per line, or piper-recording-studio "id<TAB>text"
os.environ.setdefault("PROMPTS_DIR",  "")   # a folder of .txt files, each one sentence per line
os.environ.setdefault("PROMPTS_CSV",  "")   # a CSV/TSV with a text column (LJSpeech "id|text" also read)
os.environ.setdefault("CSV_TEXT_COL", "text")
os.environ.setdefault("CSV_SEP",      "|")

# ──────────────────────────────────────────────
# DONOR  (Stage 2, pipeline step 2a)
#   Default: OmniVoice, one fixed voice for the whole dataset. Set REF_AUDIO (with an optional
#   REF_TEXT) to clone a clip, or VOICE_PROMPT to design the voice; set neither for the default design.
#   DONOR_MODEL replaces OmniVoice with a phoonnx catalog voice id or an OVOS TTS plugin name.
# ──────────────────────────────────────────────
os.environ.setdefault("REF_AUDIO",    "")    # path or URL of the clip to clone
os.environ.setdefault("REF_TEXT",     "")    # transcript of REF_AUDIO
os.environ.setdefault("VOICE_PROMPT", "")    # voice description for voice design
os.environ.setdefault("DONOR_MODEL",  "")    # phoonnx catalog voice id, or an OVOS TTS plugin name
os.environ.setdefault("SEED",         "42")
os.environ.setdefault("BATCH_SIZE",   "16")  # OmniVoice clips per call

# ──────────────────────────────────────────────
# POST-PROCESSING  (Stage 3)
# ──────────────────────────────────────────────
os.environ.setdefault("TARGET_SR",    "24000")   # final sample rate (OmniVoice writes 24 kHz)
os.environ.setdefault("TRIM_SILENCE", "1")       # 1 = trim leading/trailing silence
os.environ.setdefault("TRIM_TOP_DB",  "30")      # dB threshold for silence trimming
os.environ.setdefault("TARGET_DBFS",  "-20.0")   # loudness normalisation target

# Optional super-resolution (audiosronnx). Empty SR_ENGINE = off, plain resample only.
os.environ.setdefault("SR_ENGINE",      "")      # audiosronnx load_sr() engine alias; audiosronnx.available_models() lists them
os.environ.setdefault("SR_CACHE_DIR",   "")      # where audiosronnx caches downloaded weights; empty = its own default
os.environ.setdefault("SR_REVISION",    "")      # pin a Hugging Face revision for the SR_ENGINE weights
os.environ.setdefault("ONNX_PROVIDERS", "auto")  # onnxruntime execution providers for SR_ENGINE: comma list, or auto

TTS_LANG = _env("TTS_LANG").strip()
LANG_PRIMARY = TTS_LANG.replace("_", "-").split("-")[0].lower()
DONOR_MODEL = _env("DONOR_MODEL").strip()
REF_AUDIO, REF_TEXT, VOICE_PROMPT = _env("REF_AUDIO").strip(), _env("REF_TEXT").strip(), _env("VOICE_PROMPT").strip()

if DONOR_MODEL:
    DONOR_MODE = "model"
elif SMOKE:
    DONOR_MODE = "ci"
else:
    DONOR_MODE = "omnivoice"

if DONOR_MODE != "omnivoice" and (REF_AUDIO or REF_TEXT or VOICE_PROMPT):
    raise RuntimeError("REF_AUDIO, REF_TEXT and VOICE_PROMPT choose the OmniVoice voice; they do nothing "
                       "with DONOR_MODEL or SMOKE=True. Unset them or unset DONOR_MODEL.")
if REF_AUDIO and VOICE_PROMPT:
    raise RuntimeError("Set REF_AUDIO (clone a clip) or VOICE_PROMPT (design a voice), not both.")
if REF_AUDIO and not REF_TEXT:
    raise RuntimeError("REF_AUDIO needs REF_TEXT, the exact transcript of the clip. The notebook does not "
                       "transcribe audio: without a transcript OmniVoice would load a Whisper model.")
if REF_TEXT and not REF_AUDIO:
    raise RuntimeError("REF_TEXT is the transcript of REF_AUDIO; set REF_AUDIO too.")

print("Configuration loaded.")
print(f"  TTS_LANG   : {TTS_LANG}")
print(f"  OUTPUT_DIR : {_env('OUTPUT_DIR')}")
if DONOR_MODE == "omnivoice":
    print(f"  Donor      : OmniVoice, voice = "
          f"{'clone of ' + REF_AUDIO if REF_AUDIO else ('design: ' + VOICE_PROMPT if VOICE_PROMPT else 'default design')}")
elif DONOR_MODE == "model":
    print(f"  Donor      : DONOR_MODEL={DONOR_MODEL}")
else:
    print("  Donor      : espeak-ng (the CI path only; not a donor to build a dataset from)")
print(f"  Super-res  : {_env('SR_ENGINE') or ('off (plain resample to ' + _env('TARGET_SR') + ' Hz)')}")

---
## 1 · Load Prompts

Reads text from whichever prompts source is configured, or the built-in demo prompts when none is,
and refuses a row whose text is empty, carries a bracketed annotation (`[...]` or `<...>`), or has
no letter — the same three rules [tts_corpus_ingest.ipynb](tts_corpus_ingest.ipynb) applies to a
corpus. A refusal is counted by its reason; nothing here rewrites a prompt's text.

| Variable | Meaning |
|---|---|
| `PROMPTS_FILE` | One sentence per line, or piper-recording-studio `id<TAB>text` |
| `PROMPTS_DIR` | A folder of `.txt` files, each one sentence per line |
| `PROMPTS_CSV`, `CSV_TEXT_COL`, `CSV_SEP` | A CSV/TSV and its text column |
| none of these | Twelve built-in sentences in `TTS_LANG` (English, Spanish, Portuguese, French, German or Italian); `N_SAMPLES` of them are used |

In [ ]:
DEMO_PROMPTS = {
    "en": [
        "The old lighthouse keeper climbed the stairs every evening to light the lamp.",
        "Please remind me to water the plants before I leave for the weekend.",
        "A gentle breeze moved through the orchard as the sun slipped behind the hills.",
        "The train to the coast leaves at a quarter past eight from platform four.",
        "She opened the window and listened to the rain tapping on the roof.",
        "Could you turn the volume down a little, the baby is finally asleep?",
        "Fresh bread, warm butter and a cup of tea made the morning feel slow and kind.",
        "The museum is open every day except Monday, and children enter for free.",
        "He carefully folded the map and tucked it into the pocket of his coat.",
        "Open source software belongs to everyone who takes the time to improve it.",
        "After a long walk along the river, we stopped at a small cafe for lunch.",
        "The weather tomorrow will be cloudy in the morning, with sunshine after noon.",
    ],
    "es": [
        "El viejo farero subía las escaleras cada tarde para encender la lámpara.",
        "Por favor, recuérdame regar las plantas antes de irme el fin de semana.",
        "Una brisa suave recorría el huerto mientras el sol se escondía tras las colinas.",
        "El tren hacia la costa sale a las ocho y cuarto desde el andén cuatro.",
        "Ella abrió la ventana y escuchó la lluvia golpeando el tejado.",
        "¿Podrías bajar un poco el volumen? El bebé por fin se ha dormido.",
        "Pan fresco, mantequilla tibia y una taza de té hacían la mañana lenta y amable.",
        "El museo abre todos los días menos los lunes, y los niños entran gratis.",
        "Dobló con cuidado el mapa y lo guardó en el bolsillo del abrigo.",
        "El software libre pertenece a todos los que dedican tiempo a mejorarlo.",
        "Después de un largo paseo junto al río, paramos en un café pequeño para comer.",
        "Mañana estará nublado por la mañana, con sol después del mediodía.",
    ],
    "pt": [
        "O velho faroleiro subia as escadas todas as tardes para acender a lâmpada.",
        "Por favor, lembra-me de regar as plantas antes de sair no fim de semana.",
        "Uma brisa suave passava pelo pomar enquanto o sol se escondia atrás das colinas.",
        "O comboio para a costa parte às oito e um quarto da linha quatro.",
        "Ela abriu a janela e ficou a ouvir a chuva a bater no telhado.",
        "Podes baixar um pouco o volume? O bebé finalmente adormeceu.",
        "Pão fresco, manteiga morna e uma chávena de chá deixavam a manhã lenta e simpática.",
        "O museu abre todos os dias, exceto à segunda-feira, e as crianças entram de graça.",
        "Dobrou com cuidado o mapa e guardou-o no bolso do casaco.",
        "O software livre pertence a todos os que dedicam tempo a melhorá-lo.",
        "Depois de uma longa caminhada junto ao rio, parámos num café pequeno para almoçar.",
        "Amanhã o tempo estará nublado de manhã, com sol depois do meio-dia.",
    ],
    "fr": [
        "Le vieux gardien du phare montait l'escalier chaque soir pour allumer la lampe.",
        "Rappelle-moi d'arroser les plantes avant de partir pour le week-end, s'il te plaît.",
        "Une brise légère traversait le verger pendant que le soleil disparaissait derrière les collines.",
        "Le train pour la côte part à huit heures quinze du quai numéro quatre.",
        "Elle ouvrit la fenêtre et écouta la pluie tambouriner sur le toit.",
        "Pourrais-tu baisser un peu le volume ? Le bébé vient enfin de s'endormir.",
        "Du pain frais, du beurre tiède et une tasse de thé rendaient le matin lent et doux.",
        "Le musée est ouvert tous les jours sauf le lundi, et l'entrée est gratuite pour les enfants.",
        "Il plia soigneusement la carte et la glissa dans la poche de son manteau.",
        "Les logiciels libres appartiennent à tous ceux qui prennent le temps de les améliorer.",
        "Après une longue promenade au bord de la rivière, nous nous sommes arrêtés dans un petit café pour déjeuner.",
        "Demain, le ciel sera nuageux le matin, avec du soleil l'après-midi.",
    ],
    "de": [
        "Der alte Leuchtturmwärter stieg jeden Abend die Treppe hinauf, um die Lampe anzuzünden.",
        "Bitte erinnere mich daran, die Pflanzen zu gießen, bevor ich am Wochenende wegfahre.",
        "Eine sanfte Brise zog durch den Obstgarten, während die Sonne hinter den Hügeln versank.",
        "Der Zug zur Küste fährt um Viertel nach acht von Gleis vier ab.",
        "Sie öffnete das Fenster und hörte dem Regen auf dem Dach zu.",
        "Könntest du die Lautstärke etwas senken? Das Baby ist endlich eingeschlafen.",
        "Frisches Brot, warme Butter und eine Tasse Tee machten den Morgen langsam und freundlich.",
        "Das Museum ist jeden Tag außer montags geöffnet, und Kinder haben freien Eintritt.",
        "Er faltete die Karte sorgfältig zusammen und steckte sie in die Manteltasche.",
        "Freie Software gehört allen, die sich die Zeit nehmen, sie zu verbessern.",
        "Nach einem langen Spaziergang am Fluss machten wir in einem kleinen Café Mittagspause.",
        "Morgen ist es vormittags bewölkt, am Nachmittag scheint die Sonne.",
    ],
    "it": [
        "Il vecchio guardiano del faro saliva le scale ogni sera per accendere la lampada.",
        "Per favore, ricordami di annaffiare le piante prima di partire per il fine settimana.",
        "Una brezza leggera attraversava il frutteto mentre il sole scompariva dietro le colline.",
        "Il treno per la costa parte alle otto e un quarto dal binario quattro.",
        "Lei aprì la finestra e ascoltò la pioggia che batteva sul tetto.",
        "Potresti abbassare un po' il volume? Il bambino si è finalmente addormentato.",
        "Pane fresco, burro caldo e una tazza di tè rendevano la mattina lenta e gentile.",
        "Il museo è aperto tutti i giorni tranne il lunedì, e i bambini entrano gratis.",
        "Piegò con cura la mappa e la infilò nella tasca del cappotto.",
        "Il software libero appartiene a tutti coloro che dedicano tempo a migliorarlo.",
        "Dopo una lunga passeggiata lungo il fiume, ci siamo fermati in un piccolo bar per pranzo.",
        "Domani il cielo sarà nuvoloso al mattino, con il sole nel pomeriggio.",
    ],
}


def _clean_line(text):
    return re.sub(r"\s+", " ", text.strip())


def _read_lines_file(path):
    """One sentence per line, or piper-recording-studio 'id<TAB>text'."""
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if "\t" in line:
                line = line.split("\t", 1)[1]
            lines.append(_clean_line(line))
    return lines


def _read_lines_dir(folder):
    lines = []
    for txt in sorted(Path(folder).rglob("*.txt")):
        lines.extend(_read_lines_file(txt))
    return lines


def _read_lines_csv(path, text_col, sep):
    import csv
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        sample = f.read(4096)
        f.seek(0)
        has_header = bool(sample.strip()) and csv.Sniffer().has_header(sample)
        reader = csv.reader(f, delimiter=sep)
        if has_header:
            header = [h.strip().lower() for h in next(reader)]
            col_idx = header.index(text_col.lower()) if text_col.lower() in header else min(1, len(header) - 1)
        else:
            col_idx = 1 if sep == "|" else 0
        for row in reader:
            if len(row) > col_idx:
                lines.append(_clean_line(row[col_idx]))
    return lines


_BRACKET_RE = re.compile(r"\[[^\]]*\]|<[^>]*>")


def _text_refusal_reason(text):
    if not text or not text.strip():
        return "empty"
    if _BRACKET_RE.search(text):
        return "bracketed_annotation"
    if not any(ch.isalpha() for ch in text):
        return "no_letter"
    return None


def _demo_prompt_lines(lang, n):
    if lang not in DEMO_PROMPTS:
        raise RuntimeError(
            f"No prompts source is set and the built-in demo prompts cover only "
            f"{', '.join(sorted(DEMO_PROMPTS))}, not {lang!r}. Set PROMPTS_FILE, PROMPTS_DIR or PROMPTS_CSV."
        )
    return DEMO_PROMPTS[lang][:max(1, min(int(n), len(DEMO_PROMPTS[lang])))]


csv_path, dir_path, file_path = _env("PROMPTS_CSV"), _env("PROMPTS_DIR"), _env("PROMPTS_FILE")
n_set = sum(1 for p in (csv_path, dir_path, file_path) if p)
if n_set > 1:
    raise RuntimeError("Set at most one of PROMPTS_FILE, PROMPTS_DIR and PROMPTS_CSV.")

if csv_path:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"PROMPTS_CSV={csv_path!r} does not exist")
    raw_lines = _read_lines_csv(csv_path, _env("CSV_TEXT_COL", "text"), _env("CSV_SEP", "|"))
    prompts_source_kind = "csv"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(csv_path).stem
    prompts_source_path = csv_path
elif dir_path:
    if not os.path.isdir(dir_path):
        raise FileNotFoundError(f"PROMPTS_DIR={dir_path!r} is not a directory")
    raw_lines = _read_lines_dir(dir_path)
    prompts_source_kind = "dir"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(dir_path).name
    prompts_source_path = dir_path
elif file_path:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"PROMPTS_FILE={file_path!r} does not exist")
    raw_lines = _read_lines_file(file_path)
    prompts_source_kind = "file"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(file_path).stem
    prompts_source_path = file_path
else:
    raw_lines = _demo_prompt_lines(LANG_PRIMARY, N_SAMPLES)
    prompts_source_kind = "demo"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or f"demo-{LANG_PRIMARY}"
    prompts_source_path = None
    print(f"No prompt source configured — using {len(raw_lines)} built-in demo prompts ({LANG_PRIMARY}).")
    if SMOKE:
        # The CI fixture exists to show refusal working, not merely to run:
        # a rule that stopped refusing must fail this cell, not print a quiet count.
        raw_lines = raw_lines + [
            "[inaudible]",           # refused: bracketed annotation
            "12345",                 # refused: no letter
        ]

reason_counts = Counter()
PROMPT_ROWS = []
for text in raw_lines:
    reason = _text_refusal_reason(text)
    if reason:
        reason_counts[reason] += 1
        continue
    PROMPT_ROWS.append({"id": f"{prompts_source_name}:{len(PROMPT_ROWS):06d}", "text": text})
    reason_counts["passed"] += 1

print(f"Prompt lines read : {len(raw_lines)}")
print("Text validation:")
for reason in ("empty", "bracketed_annotation", "no_letter", "passed"):
    print(f"  {reason:22s} {reason_counts.get(reason, 0)}")
if not PROMPT_ROWS:
    raise RuntimeError("Every prompt was refused, or none were loaded; nothing to synthesise.")

if SMOKE and prompts_source_kind == "demo":
    assert reason_counts.get("bracketed_annotation", 0) >= 1, (
        "the CI fixture's bracketed-annotation prompt was not refused"
    )
    assert reason_counts.get("no_letter", 0) >= 1, (
        "the CI fixture's letterless prompt was not refused"
    )

---
## 2 · Donor Synthesis

**OmniVoice (the default).** Loads `k2-fsa/OmniVoice` on the GPU in fp16, checks that `TTS_LANG`
is one of its languages, and fixes one voice for the whole run:

- `REF_AUDIO` set: the clip is the voice and `REF_TEXT`, its transcript, is required. The notebook never transcribes audio.
- `VOICE_PROMPT` set, or nothing set: OmniVoice designs the voice once, from `VOICE_PROMPT` or from the
  default `female, young adult, moderate pitch` (plus `american accent` for English). Voice design
  draws a new speaker on every call, so the notebook speaks the first prompts once, keeps that clip
  as `voice/anchor.wav`, and clones it for every row. Every row therefore has the same speaker. To
  add rows to the dataset later in the same voice, set `REF_AUDIO` to `voice/anchor.wav` and
  `REF_TEXT` to the content of `voice/anchor.txt`.

A language OmniVoice does not support stops the notebook with a message that names `DONOR_MODEL`.

**`DONOR_MODEL`.** A plain string: a voice id from the phoonnx catalog, or an OVOS TTS plugin name
(the notebook installs the plugin from PyPI when it is missing). Quality is the caller's choice: a
small VITS or Piper-class voice makes a weak training set. The voices of the published dataset
identities are not donors, and the notebook refuses them.

**`SMOKE=True`.** The CI path: `espeak-ng` on a CPU. It checks that the notebook runs and fails when
a prompt produces no audio.

In [ ]:
import json
import shlex
import shutil
import subprocess
import sys
import time

_pkgs = ['"librosa>=0.10.2"', '"soundfile>=0.14.0"', '"numpy>=2.0"', '"tqdm>=4.70.1"']
if DONOR_MODE == "omnivoice":
    _pkgs.append('"omnivoice>=0.2.1"')
elif DONOR_MODE == "model":
    _pkgs += ['"ovos-plugin-manager>=2.12.4a1"', '"phoonnx"', '"nest-asyncio>=1.6.0"']
%pip install --quiet --pre {" ".join(_pkgs)}

import numpy as np
import soundfile as sf
from tqdm import tqdm

raw_dir = Path(_env("OUTPUT_DIR")) / "donor_raw"
raw_dir.mkdir(parents=True, exist_ok=True)
voice_dir = Path(_env("OUTPUT_DIR")) / "voice"

# What the donor stage fills in for the manifest and the sidecar.
DONOR_PLUGIN_ID = DONOR_VOICE_ID = DONOR_VERSION = DONOR_NOTE = None
DONOR_DETAILS = {}
GENERATION = {}
SYNTH_ROWS = []
n_synth_failed = 0

# The voices of the published dataset identities, which are conversion targets and never donors.
_REFUSED_VOICE_RE = re.compile(r"^openvoiceos/.*[_-](miro|dii)(?:[_-]|$)")


def _usable(audio):
    """False for an empty, non-finite or silent waveform."""
    audio = np.asarray(audio)
    return bool(audio.size) and bool(np.isfinite(audio).all()) and float(np.abs(audio).max()) > 1e-4


def _gpu_memory():
    import torch
    out = {
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2),
        "total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    }
    try:
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True, check=True).stdout.split()[0]
        out["nvidia_smi_used_gib"] = round(int(used) / 1024, 2)
    except (OSError, subprocess.CalledProcessError, IndexError, ValueError):
        pass
    return out


def _write_raw(row, audio, sr):
    path = raw_dir / f"{_slug(row['id'])}.wav"
    sf.write(str(path), np.asarray(audio, dtype=np.float32), sr)
    return path


if DONOR_MODE == "omnivoice":
    import torch
    from importlib.metadata import version
    from omnivoice import OmniVoice
    from omnivoice.utils.lang_map import LANG_IDS, LANG_NAME_TO_ID

    def _omnivoice_language(tag):
        primary = tag.replace("_", "-").split("-")[0].lower()
        if primary in LANG_IDS:
            return primary
        return LANG_NAME_TO_ID.get(tag.strip().lower())

    OMNI_LANG = _omnivoice_language(TTS_LANG)
    if OMNI_LANG is None:
        raise RuntimeError(
            f"OmniVoice does not support TTS_LANG={TTS_LANG!r}. It speaks {len(LANG_IDS)} languages "
            f"(omnivoice.utils.lang_map.LANG_NAME_TO_ID lists their names and codes; a variety is named by its "
            f"own code or name, for example arb or 'standard arabic'). If the language is not on the list, "
            f"set DONOR_MODEL to a model from the phoonnx catalog or to the name of an OVOS TTS plugin."
        )

    try:
        (torch.ones(8, device="cuda") * 2).sum().item()
    except Exception as exc:
        raise RuntimeError(
            f"OmniVoice needs a GPU and no GPU can run a calculation here ({exc}). On Kaggle: Settings, "
            f"Accelerator, GPU T4. For a CPU-only run set DONOR_MODEL."
        ) from exc

    seed = int(_env("SEED", "42"))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.reset_peak_memory_stats()
    gpu_name = torch.cuda.get_device_name(0)

    t0 = time.time()
    tts = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
    load_seconds = time.time() - t0
    gpu_after_load = _gpu_memory()
    print(f"OmniVoice {version('omnivoice')} loaded on {gpu_name} in {load_seconds:.0f} s "
          f"({gpu_after_load['peak_allocated_gib']} GiB allocated)")

    voice_dir.mkdir(parents=True, exist_ok=True)
    if REF_AUDIO:
        assert REF_TEXT, "REF_TEXT is required with REF_AUDIO"   # a missing transcript would make OmniVoice load its Whisper model
        if re.match(r"^https?://", REF_AUDIO):
            import urllib.request
            ref_path = voice_dir / ("reference" + (Path(REF_AUDIO.split("?")[0]).suffix or ".wav"))
            urllib.request.urlretrieve(REF_AUDIO, ref_path)
        else:
            ref_path = Path(REF_AUDIO)
            if not ref_path.is_file():
                raise FileNotFoundError(f"REF_AUDIO={REF_AUDIO!r} does not exist")
        clone_prompt = tts.create_voice_clone_prompt(ref_audio=str(ref_path), ref_text=REF_TEXT)
        voice_mode = "clone"
        DONOR_VOICE_ID = f"clone:{ref_path.name}"
        voice_detail = {"ref_audio": ref_path.name, "ref_text": clone_prompt.ref_text}
    else:
        design = VOICE_PROMPT or "female, young adult, moderate pitch" + (", american accent" if OMNI_LANG == "en" else "")
        # A passage of the first prompts, long enough for a clean 5 to 12 s reference.
        anchor_text = ""
        for _row in PROMPT_ROWS:
            anchor_text = (anchor_text + " " + _row["text"]).strip()
            if len(anchor_text) >= 100:
                break
        anchor_text = anchor_text[:300]
        anchor = tts.generate(text=anchor_text, language=OMNI_LANG, instruct=design)[0]
        if not _usable(anchor):
            raise RuntimeError(f"OmniVoice produced no audio for the voice design {design!r}.")
        sf.write(str(voice_dir / "anchor.wav"), np.asarray(anchor, dtype=np.float32), tts.sampling_rate)
        (voice_dir / "anchor.txt").write_text(anchor_text, encoding="utf-8")
        clone_prompt = tts.create_voice_clone_prompt(
            ref_audio=(torch.from_numpy(np.asarray(anchor, dtype=np.float32)).unsqueeze(0), tts.sampling_rate),
            ref_text=anchor_text,
        )
        voice_mode = "design"
        DONOR_VOICE_ID = f"design:{design}"
        voice_detail = {"voice_prompt": design, "anchor_seconds": round(len(anchor) / tts.sampling_rate, 2),
                        "anchor_text_chars": len(anchor_text)}
    print(f"Voice: {DONOR_VOICE_ID}")

    def _generate(texts):
        # A batch with one degenerate item is retried item by item; an item that still fails is None.
        try:
            return tts.generate(text=texts, language=OMNI_LANG, voice_clone_prompt=clone_prompt)
        except Exception:
            out = []
            for t in texts:
                try:
                    out.append(tts.generate(text=[t], language=OMNI_LANG, voice_clone_prompt=clone_prompt)[0])
                except Exception:
                    out.append(None)
            return out

    batch_size = max(1, int(_env("BATCH_SIZE", "16")))
    torch.cuda.synchronize()
    gen_seconds = 0.0
    audio_seconds = 0.0
    for start in tqdm(range(0, len(PROMPT_ROWS), batch_size), desc="OmniVoice batches", unit="batch", file=sys.stdout):
        batch = PROMPT_ROWS[start:start + batch_size]
        t1 = time.time()
        out = _generate([r["text"] for r in batch])
        torch.cuda.synchronize()
        gen_seconds += time.time() - t1
        for row, audio in zip(batch, out):
            if audio is None or not _usable(audio):
                n_synth_failed += 1
                tqdm.write(f"  Failed {row['id']}: no usable audio")
                continue
            audio_seconds += len(audio) / tts.sampling_rate
            SYNTH_ROWS.append({**row, "raw_audio": str(_write_raw(row, audio, tts.sampling_rate))})

    DONOR_PLUGIN_ID = "omnivoice"
    DONOR_VERSION = version("omnivoice")
    DONOR_NOTE = f"OmniVoice (k2-fsa/OmniVoice, fp16, {gpu_name}), voice mode {voice_mode}, language {OMNI_LANG}"
    DONOR_DETAILS = {"model": "k2-fsa/OmniVoice", "dtype": "float16", "language": OMNI_LANG,
                     "voice_mode": voice_mode, "seed": seed, "batch_size": batch_size, **voice_detail}
    clone_prompt.save(str(voice_dir / "voice_clone_prompt.pt"))
    GENERATION = {
        "device": gpu_name,
        "clips": len(SYNTH_ROWS),
        "seconds": round(gen_seconds, 2),
        "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None,
        "audio_seconds": round(audio_seconds, 2),
        "audio_s_per_s": round(audio_seconds / gen_seconds, 2) if gen_seconds else None,
        "model_load_seconds": round(load_seconds, 1),
        "gpu_memory": _gpu_memory(),
    }
    del tts, clone_prompt
    torch.cuda.empty_cache()

elif DONOR_MODE == "model":
    if _REFUSED_VOICE_RE.search(DONOR_MODEL.lower()):
        raise RuntimeError(f"DONOR_MODEL={DONOR_MODEL!r} is the voice of a published dataset identity, "
                           f"which is a conversion target and never a donor.")
    import nest_asyncio
    nest_asyncio.apply()
    from ovos_plugin_manager.tts import load_tts_plugin
    from phoonnx.model_manager import TTSModelManager

    catalog = TTSModelManager()
    catalog.load()
    catalog.merge_default_voices()
    if DONOR_MODEL in catalog.voices:
        DONOR_PLUGIN_ID, DONOR_VOICE_ID = "ovos-tts-plugin-phoonnx", DONOR_MODEL
        plugin_config = {"lang": TTS_LANG, "voice": DONOR_MODEL}
        DONOR_NOTE = f"phoonnx catalog voice {DONOR_MODEL!r}"
    else:
        if DONOR_MODEL in ("ovos-tts-plugin-phoonnx", "phoonnx"):
            raise RuntimeError("DONOR_MODEL names the phoonnx plugin without a voice. Name one voice id from "
                               "the phoonnx catalog (phoonnx-voices list-voices --lang <tag>).")
        DONOR_PLUGIN_ID, DONOR_VOICE_ID = DONOR_MODEL, ""
        plugin_config = {"lang": TTS_LANG}
        DONOR_NOTE = f"OVOS TTS plugin {DONOR_MODEL!r} with its default voice"
    print(f"Donor: {DONOR_NOTE}")

    donor_cls = load_tts_plugin(DONOR_PLUGIN_ID)
    if donor_cls is None:
        print(f"{DONOR_PLUGIN_ID!r} is not an installed OVOS TTS plugin; installing it from PyPI.")
        get_ipython().run_line_magic("pip", f"install --quiet --pre {shlex.quote(DONOR_PLUGIN_ID)}")
        import importlib
        importlib.invalidate_caches()
        donor_cls = load_tts_plugin(DONOR_PLUGIN_ID)
    if donor_cls is None:
        raise RuntimeError(f"DONOR_MODEL={DONOR_MODEL!r} is neither a phoonnx catalog voice id nor an OVOS "
                           f"TTS plugin that pip can install. If the plugin was just installed, restart the "
                           f"kernel and run again.")
    donor_plugin = donor_cls(config=plugin_config)
    DONOR_VERSION = None
    import importlib.metadata
    for _candidate in (donor_cls.__module__.split(".")[0], DONOR_PLUGIN_ID, DONOR_PLUGIN_ID.replace("_", "-")):
        try:
            DONOR_VERSION = importlib.metadata.version(_candidate)
            break
        except importlib.metadata.PackageNotFoundError:
            continue
    # The plugin's own declared format, never assumed to be WAV.
    donor_audio_ext = (getattr(donor_plugin, "audio_ext", None) or "wav").lstrip(".")
    print(f"Loaded donor plugin {DONOR_PLUGIN_ID!r} (package version {DONOR_VERSION or 'unknown'}, "
          f"audio_ext={donor_audio_ext!r})")

    t0 = time.time()
    for row in tqdm(PROMPT_ROWS, desc="Donor synthesis", unit="utt", file=sys.stdout):
        raw_path = raw_dir / f"{_slug(row['id'])}.{donor_audio_ext}"
        try:
            donor_plugin.get_tts(row["text"], str(raw_path), lang=TTS_LANG, voice=DONOR_VOICE_ID or None)
            if not raw_path.is_file() or raw_path.stat().st_size == 0:
                raise RuntimeError("the plugin wrote no audio")
            SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
        except Exception as exc:
            n_synth_failed += 1
            tqdm.write(f"  Failed {row['id']}: {exc}")
    gen_seconds = time.time() - t0
    GENERATION = {"device": "plugin default", "clips": len(SYNTH_ROWS), "seconds": round(gen_seconds, 2),
                  "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None}

else:
    # CI path only. espeak-ng needs no network and no GPU; it is not a donor to build a dataset from.
    espeak = shutil.which("espeak-ng") or shutil.which("espeak")
    if not espeak:
        raise RuntimeError("SMOKE=True speaks with espeak-ng, which is not installed "
                           "(Linux: apt-get install espeak-ng).")
    DONOR_PLUGIN_ID, DONOR_VOICE_ID = "espeak-ng", LANG_PRIMARY
    _ver = re.search(r"\d+(?:\.\d+)+", subprocess.run([espeak, "--version"], capture_output=True, text=True).stdout)
    DONOR_VERSION = _ver.group(0) if _ver else None
    DONOR_NOTE = "espeak-ng, the CI path only: a CPU synthesiser that checks the notebook runs, not a donor for a dataset"
    print(f"Donor: {DONOR_NOTE}")
    t0 = time.time()
    for row in tqdm(PROMPT_ROWS, desc="Donor synthesis", unit="utt", file=sys.stdout):
        raw_path = raw_dir / f"{_slug(row['id'])}.wav"
        try:
            subprocess.run([espeak, "-v", LANG_PRIMARY, "-w", str(raw_path), row["text"]],
                           check=True, capture_output=True)
            audio, _sr = sf.read(str(raw_path), dtype="float32")
            if not _usable(audio):
                raise RuntimeError("espeak-ng wrote silence")
            SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
        except Exception as exc:
            n_synth_failed += 1
            tqdm.write(f"  Failed {row['id']}: {exc}")
    gen_seconds = time.time() - t0
    GENERATION = {"device": "cpu", "clips": len(SYNTH_ROWS), "seconds": round(gen_seconds, 2),
                  "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None}

print(f"Synthesised {len(SYNTH_ROWS)} of {len(PROMPT_ROWS)} prompt(s); {n_synth_failed} failed")
if GENERATION:
    print(f"Generation: {json.dumps(GENERATION)}")
if not SYNTH_ROWS:
    raise RuntimeError("Donor synthesis produced no audio.")
if SMOKE and len(SYNTH_ROWS) != len(PROMPT_ROWS):
    raise RuntimeError(f"The CI path synthesised {len(SYNTH_ROWS)} of {len(PROMPT_ROWS)} prompts; every one must succeed.")

---
## 3 · Post-Processing

For every synthesised clip: an optional `audiosronnx` super-resolution pass, resample to
`TARGET_SR`, downmix to mono, trim leading/trailing silence, and normalise loudness to
`TARGET_DBFS`. `SR_ENGINE` empty (the default) skips super-resolution entirely — plain resampling
is used. Every `audiosronnx` engine's constructor is read through `inspect.signature` before use:
only the options in `SR_<OPTION>` env vars that the selected engine actually accepts are read,
and setting one it does not accept stops the notebook, naming the engine and the option (the same
pattern [tts_enhance.ipynb](tts_enhance.ipynb) uses for its own `ENHANCE_<OPTION>`).

| Variable | Meaning |
|---|---|
| `TARGET_SR` | Final sample rate |
| `TRIM_SILENCE`, `TRIM_TOP_DB` | Silence trimming |
| `TARGET_DBFS` | Loudness normalisation target |
| `SR_ENGINE` | `audiosronnx` `load_sr()` engine alias; empty = off. `audiosronnx.available_models()` lists them |
| `SR_<OPTION>` | One env var per constructor option of the selected `SR_ENGINE` |
| `SR_CACHE_DIR`, `SR_REVISION` | Weight cache directory and Hugging Face revision pin |
| `ONNX_PROVIDERS` | onnxruntime execution providers for `SR_ENGINE`: comma list, or `auto` |

In [ ]:
sr_engine = _env("SR_ENGINE").strip().lower()
if sr_engine:
    %pip install --quiet --pre "audiosronnx>=0.9.0a2" "onnxruntime>=1.30.0"
else:
    print("SR_ENGINE not set — no super-resolution stage, plain resampling only.")

In [ ]:
import numpy as np
import soundfile as sf
import librosa

target_sr = int(_env("TARGET_SR", "24000"))
trim_silence = _flag("TRIM_SILENCE")
trim_top_db = int(_env("TRIM_TOP_DB", "30"))
target_dbfs = float(_env("TARGET_DBFS", "-20.0"))

sr_model = None
if sr_engine:
    import inspect
    from audiosronnx import load_sr
    from audiosronnx.base import get_engine

    def _parse_env_value(raw):
        low = raw.strip().lower()
        if low in ("true", "false"):
            return low == "true"
        for caster in (int, float):
            try:
                return caster(raw)
            except ValueError:
                pass
        return raw

    def _onnx_providers():
        raw = _env("ONNX_PROVIDERS", "auto").strip()
        if raw.lower() != "auto":
            return [p.strip() for p in raw.split(",") if p.strip()]
        import onnxruntime as ort
        priority = ["CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider", "CPUExecutionProvider"]
        available = set(ort.get_available_providers())
        ordered = [p for p in priority if p in available]
        if "CPUExecutionProvider" not in ordered:
            ordered.append("CPUExecutionProvider")
        return ordered

    _entry = get_engine(sr_engine)
    _sig = inspect.signature(_entry.adapter_class.__init__)
    _accepted = set(_sig.parameters) - {"self", "providers", "cache_dir", "revision"}
    sr_kwargs = {}
    for _name in _accepted:
        _raw = _env(f"SR_{_name.upper()}")
        if _raw:
            sr_kwargs[_name] = _parse_env_value(_raw)
    for _key in os.environ:
        if _key.startswith("SR_") and _key not in {"SR_ENGINE", "SR_CACHE_DIR", "SR_REVISION"}:
            _opt = _key[len("SR_"):].lower()
            if _opt not in _accepted:
                raise RuntimeError(
                    f"{_key} is set but {sr_engine!r} has no {_opt!r} option (accepts: {sorted(_accepted)})"
                )

    sr_model = load_sr(
        sr_engine,
        providers=_onnx_providers(),
        cache_dir=_env("SR_CACHE_DIR") or None,
        revision=_env("SR_REVISION") or None,
        **sr_kwargs,
    )
    print(f"Loaded audiosronnx SR engine {sr_engine!r} with options {sr_kwargs}")


wavs_dir = Path(_env("OUTPUT_DIR")) / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

FINAL_ROWS = []
n_post_failed = 0
for row in tqdm(SYNTH_ROWS, desc="Post-processing", unit="utt", file=sys.stdout):
    out_name = f"{_slug(row['id'])}.wav"
    out_path = wavs_dir / out_name
    try:
        wav, sr = librosa.load(row["raw_audio"], sr=None, mono=True)

        if sr_model is not None:
            wav, sr = sr_model.upscale(wav, sample_rate=sr)

        if sr != target_sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
            sr = target_sr

        if trim_silence:
            wav, _ = librosa.effects.trim(wav, top_db=trim_top_db)

        if not _usable(wav):
            raise RuntimeError("the clip is empty or silent after trimming")

        rms = float(np.sqrt(np.mean(wav ** 2) + 1e-9))
        current_dbfs = 20 * np.log10(rms + 1e-9)
        gain = 10 ** ((target_dbfs - current_dbfs) / 20.0)
        wav = np.clip(wav * gain, -1.0, 1.0).astype(np.float32)

        sf.write(str(out_path), wav, sr)
        FINAL_ROWS.append({
            **row,
            "audio": f"wavs/{out_name}",
            "duration_s": round(len(wav) / sr, 3),
            "sample_rate": int(sr),
        })
    except Exception as exc:
        n_post_failed += 1
        tqdm.write(f"  Failed to post-process {row['id']}: {exc}")

print(f"Post-processed {len(FINAL_ROWS)} of {len(SYNTH_ROWS)} clip(s); {n_post_failed} failed")
if not FINAL_ROWS:
    raise RuntimeError("Post-processing produced no audio.")
if SMOKE and len(FINAL_ROWS) != len(SYNTH_ROWS):
    raise RuntimeError(f"The CI path post-processed {len(FINAL_ROWS)} of {len(SYNTH_ROWS)} clips; every one must succeed.")

---
## 4 · Manifest, Sidecar and Summary

In [ ]:
import csv

manifest_rows = []
for row in FINAL_ROWS:
    manifest_rows.append({
        "id": row["id"],
        "text": row["text"],
        "lang": TTS_LANG,
        "audio": row["audio"],
        "duration_s": row["duration_s"],
        "sample_rate": row["sample_rate"],
        "source": {
            "kind": "tts",
            "plugin": DONOR_PLUGIN_ID,
            "voice": DONOR_VOICE_ID,
            "version": DONOR_VERSION,
        },
    })

out_dir = Path(_env("OUTPUT_DIR"))
manifest_path = out_dir / _env("MANIFEST_NAME", "manifest.jsonl")
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

metadata_csv_path = out_dir / "metadata.csv"
with open(metadata_csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    for row in manifest_rows:
        writer.writerow([Path(row["audio"]).stem, row["text"]])

meta = {
    "notebooks": ["tts_dataset_gen"],
    "prompts_source": {
        "kind": prompts_source_kind,
        "name": prompts_source_name,
        "path": os.path.basename(prompts_source_path) if prompts_source_path else None,
    },
    "donor": {
        "plugin": DONOR_PLUGIN_ID,
        "voice": DONOR_VOICE_ID,
        "version": DONOR_VERSION,
        "mode": DONOR_MODE,
        "ci_path_only": DONOR_MODE == "ci",
        "note": DONOR_NOTE,
        **DONOR_DETAILS,
    },
    "generation": GENERATION,
    "counts": {
        "prompt_lines_read": len(raw_lines),
        "prompts_refused": {k: v for k, v in reason_counts.items() if k != "passed"},
        "prompts_passed_text_validation": reason_counts.get("passed", 0),
        "donor_synthesis_failed": n_synth_failed,
        "post_processing_failed": n_post_failed,
        "rows_written": len(manifest_rows),
        "hours_written": round(sum(r["duration_s"] for r in manifest_rows) / 3600, 4),
    },
}
meta_path = out_dir / (_env("MANIFEST_NAME", "manifest.jsonl") + ".meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("=" * 65)
print("  OUTPUT SUMMARY")
print("=" * 65)
print(f"  Manifest     : {manifest_path}  ({len(manifest_rows)} rows)")
print(f"  Sidecar      : {meta_path}")
print(f"  metadata.csv : {metadata_csv_path}")
print(f"  Audio        : {wavs_dir}")
if GENERATION.get("clips_per_s"):
    print(f"  Generation   : {GENERATION['clips']} clips in {GENERATION['seconds']} s = {GENERATION['clips_per_s']} clips/s"
          + (f", GPU memory peak {GENERATION['gpu_memory']['peak_allocated_gib']} GiB allocated / "
             f"{GENERATION['gpu_memory']['peak_reserved_gib']} GiB reserved of {GENERATION['gpu_memory']['total_gib']} GiB"
             if "gpu_memory" in GENERATION else ""))
if manifest_rows:
    print("  First row    :", json.dumps(manifest_rows[0], ensure_ascii=False))
print("=" * 65)

---
## Next steps

This manifest feeds any of the later chain steps directly:

- [tts_voice_convert.ipynb](tts_voice_convert.ipynb) — turn this donor speech into a target voice identity
- [tts_measure.ipynb](tts_measure.ipynb) — score naturalness, speaker similarity and the recogniser round trip
- [tts_select.ipynb](tts_select.ipynb) — decide what a trainer keeps
- [tts_publish.ipynb](tts_publish.ipynb) — publish the selected rows as a Hugging Face dataset

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.